In [12]:

import pandas as pd
import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
data = load_iris()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

print("Dataset Loaded Successfully")
print("Dataset Shape:", X.shape)
print("Target Names:", data.target_names)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("\nTraining Data:", X_train.shape)
print("Validation Data:", X_val.shape)
print("Testing Data:", X_test.shape)

baseline_model = LogisticRegression(max_iter=5000)

baseline_model.fit(X_train, y_train)

y_pred = baseline_model.predict(X_val)

baseline_accuracy = accuracy_score(y_val, y_pred)

print("\nBaseline Accuracy:", baseline_accuracy)

cv_scores = cross_val_score(
    LogisticRegression(max_iter=5000),
    X_train,
    y_train,
    cv=5,
    scoring='accuracy'
)

cv_mean = cv_scores.mean()

print("\nCross Validation Scores:", cv_scores)
print("Cross Validation Mean:", cv_mean)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=5000))
])

pipeline.fit(X_train, y_train)

pipeline_pred = pipeline.predict(X_val)

pipeline_accuracy = accuracy_score(y_val, pipeline_pred)

print("\nPipeline Accuracy:", pipeline_accuracy)

param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)

print("\nBest Parameter:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

best_model = grid_search.best_estimator_

y_test_pred = best_model.predict(X_test)

tuned_accuracy = accuracy_score(y_test, y_test_pred)

print("\nTuned Test Accuracy:", tuned_accuracy)
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))
print("\nClassification Report:")
print(classification_report(
    y_test, y_test_pred,
    target_names=data.target_names
))
comparison = pd.DataFrame({
    'Model': [
        'Baseline Logistic Regression',
        '5-Fold Cross Validation',
        'Pipeline Model',
        'Tuned Logistic Regression'
    ],
    'Accuracy': [
        baseline_accuracy,
        cv_mean,
        pipeline_accuracy,
        tuned_accuracy
    ]
})
print("\nModel Comparison Table")
print(comparison)
print("\nConclusion: Model performance evaluated successfully.")
print("Baseline Accuracy:", baseline_accuracy)
print("Cross Validation Mean:", cv_mean)
print("Tuned Accuracy:", tuned_accuracy)

Dataset Loaded Successfully
Dataset Shape: (150, 4)
Target Names: ['setosa' 'versicolor' 'virginica']

Training Data: (105, 4)
Validation Data: (22, 4)
Testing Data: (23, 4)

Baseline Accuracy: 0.8636363636363636

Cross Validation Scores: [0.95238095 0.95238095 1.         0.95238095 1.        ]
Cross Validation Mean: 0.9714285714285713

Pipeline Accuracy: 0.8636363636363636

Best Parameter: {'classifier__C': 1}
Best CV Score: 0.980952380952381

Tuned Test Accuracy: 0.9565217391304348

Confusion Matrix:
[[7 0 0]
 [0 8 0]
 [0 1 7]]

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00         7
  versicolor       0.89      1.00      0.94         8
   virginica       1.00      0.88      0.93         8

    accuracy                           0.96        23
   macro avg       0.96      0.96      0.96        23
weighted avg       0.96      0.96      0.96        23


Model Comparison Table
                          Model  Ac